In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!mkdir -p "/content/hw3-data-release"

In [3]:
!tar -xf "/content/drive/MyDrive/Visual Recognition/hw3/hw3-data-release.tar" -C "/content/hw3-data-release"

In [1]:
main_data_path = "/content/hw3-data-release"
output_path =  main_data_path # "content/hw3-data-release/mytrain"

In [4]:
!pip install imagecodecs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 139.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [12]:
import albumentations as A
import numpy as np
import tifffile
import torch
import torch.nn as nn
import json
import random
import shutil
import copy
import io
import math
import time

from pathlib import Path
from pycocotools import mask as mask_utils
from skimage.measure import label as cc_label
from pycocotools.coco import COCO
from torch.utils.data import Dataset
from contextlib import redirect_stdout
from pycocotools.cocoeval import COCOeval
from torchvision.models.detection import (
    MaskRCNN_ResNet50_FPN_V2_Weights,
    maskrcnn_resnet50_fpn_v2,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader
from torchvision import ops

In [ ]:
CLASS_FILES = ["class1.tif", "class2.tif", "class3.tif", "class4.tif"]
CLASS_TO_CAT_ID = {f"class{i}": i for i in range(1, 5)}
MIN_INSTANCE_AREA = 5


def encode_binary_mask_to_rle(binary_mask):

    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


def bbox_from_binary_mask(binary_mask):
    ys, xs = np.where(binary_mask > 0)
    if ys.size == 0:
        return None
    x_min, x_max = xs.min(), xs.max()
    y_min, y_max = ys.min(), ys.max()
    return [float(x_min), float(y_min), float(x_max - x_min + 1), float(y_max - y_min + 1)]


def extract_instances_from_class_mask(mask):
    unique_vals = np.unique(mask)
    unique_nonzero = unique_vals[unique_vals != 0]
    if unique_nonzero.size == 0:
        return

    if unique_nonzero.size == 1 and unique_nonzero[0] == 1:
        labeled = cc_label(mask > 0, connectivity=1)
        for lbl in range(1, labeled.max() + 1):
            inst = (labeled == lbl)
            if inst.sum() >= MIN_INSTANCE_AREA:
                yield inst.astype(np.uint8)
    else:
        for v in unique_nonzero:
            inst = (mask == v)
            if inst.sum() >= MIN_INSTANCE_AREA:
                yield inst.astype(np.uint8)


def process_sample_folder(sample_dir, image_id, ann_start_id):

    image_path = sample_dir / "image.tif"
    if not image_path.exists():
        raise FileNotFoundError(f"Missing image.tif in {sample_dir}")

    img = tifffile.imread(str(image_path))
    if img.ndim == 2:
        height, width = img.shape
    else:
        height, width = img.shape[:2]

    image_info = {
        "id": image_id,
        "file_name": f"{sample_dir.name}.tif",
        "height": int(height),
        "width": int(width),
    }

    annotations = []
    ann_id = ann_start_id
    for cls_file in CLASS_FILES:
        cls_path = sample_dir / cls_file
        if not cls_path.exists():
            continue
        cls_mask = tifffile.imread(str(cls_path))
        if cls_mask.shape[:2] != (height, width):
            raise ValueError(
                f"Shape mismatch in {sample_dir}: image {(height, width)} vs "
                f"{cls_file} {cls_mask.shape[:2]}"
            )
        cat_id = CLASS_TO_CAT_ID[cls_file.replace(".tif", "")]
        for inst_mask in extract_instances_from_class_mask(cls_mask):
            bbox = bbox_from_binary_mask(inst_mask)
            if bbox is None:
                continue
            rle = encode_binary_mask_to_rle(inst_mask)
            area = float(inst_mask.sum())
            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": cat_id,
                "segmentation": rle,
                "bbox": bbox,
                "area": area,
                "iscrowd": 0,
            })
            ann_id += 1

    return image_info, annotations, ann_id


def build_coco_dict(images, annotations):
    return {
        "categories": [
            {"id": 1, "name": "class1"},
            {"id": 2, "name": "class2"},
            {"id": 3, "name": "class3"},
            {"id": 4, "name": "class4"},
        ],
        "images": images,
        "annotations": annotations,
    }


def stage_image(src_dir, dst_dir, link):
    src = src_dir / "image.tif"
    dst = dst_dir / f"{src_dir.name}.tif"
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    if link:
        dst.symlink_to(src.resolve())
    else:
        shutil.copy2(src, dst)

def build_split(samples, train_images_dir, image_id_start=1, ann_id_start=1):
    COPY_IMAGES = True
    images, annotations = [], []
    image_id = image_id_start
    ann_id = ann_id_start
    for s in samples:
        img_info, anns, ann_id = process_sample_folder(s, image_id, ann_id)
        images.append(img_info)
        annotations.extend(anns)
        stage_image(s, train_images_dir, link=not COPY_IMAGES)
        image_id += 1
    return images, annotations

def main():
  data_root = Path(main_data_path)
  out_root = Path(output_path)
  train_dir = data_root / "train"

  if not train_dir.is_dir():
      raise SystemExit(f"Train directory not found: {train_dir}")

  seed = 2
  val_ratio = 0.08

  sample_dirs = sorted([p for p in train_dir.iterdir() if p.is_dir()])
  print(f"Found {len(sample_dirs)} training samples in {train_dir}")

  rng = random.Random(seed)
  shuffled = sample_dirs.copy()
  rng.shuffle(shuffled)
  n_val = max(0, int(round(len(shuffled) * val_ratio)))
  val_samples = shuffled[:n_val]
  # train_samples = shuffled[n_val:]
  train_samples = shuffled
  print(f"Train: {len(train_samples)}  Val: {len(val_samples)}")

  train_images_dir = out_root / "train_images"
  ann_dir = out_root / "annotations"
  train_images_dir.mkdir(parents=True, exist_ok=True)
  ann_dir.mkdir(parents=True, exist_ok=True)

  train_images, train_anns = build_split(train_samples, train_images_dir, 1, 1)
  val_images, val_anns = build_split(val_samples,
      train_images_dir,
      image_id_start=len(train_samples) + 1,
      ann_id_start=len(train_anns) + 1,
  )

  train_coco = build_coco_dict(train_images, train_anns)
  val_coco = build_coco_dict(val_images, val_anns)

  with open(ann_dir / "train.json", "w") as f:
      json.dump(train_coco, f)
  with open(ann_dir / "val.json", "w") as f:
      json.dump(val_coco, f)

if __name__ == "__main__":
    main()



In [8]:
MIN_AREA_AFTER_AUGMENTATION = 10

def build_train_augmentation():

    return A.Compose(
        [

            A.RandomBrightnessContrast(p=0.3),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Affine(
                scale=(0.9, 1.1),
                rotate=(-15, 15),
                translate_percent=(-0.05, 0.05),
                p=0.5,
                border_mode=0,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.2, contrast_limit=0.2, p=0.5
            ),
            A.ElasticTransform(p=0.3),
            A.CLAHE(p=0.2),

        ],
        bbox_params=A.BboxParams(
            format="pascal_voc",
            label_fields=["labels"],
            min_visibility=0.1,
        ),
    )

class CocoMyDataset(Dataset):
    def __init__(self, images_dir, ann_file, transforms=None):
        self.images_dir = Path(images_dir)
        self.coco = COCO(str(ann_file))
        self.image_ids = sorted(self.coco.imgs.keys())

        self.image_ids = [
            i for i in self.image_ids
            if len(self.coco.getAnnIds(imgIds=i)) > 0
        ]
        self.transforms = transforms

    def __len__(self):
        return len(self.image_ids)

    def _load_image_uint8(self, info):
        path = self.images_dir / info["file_name"]
        img = tifffile.imread(str(path))
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=-1)
        elif img.shape[-1] == 4:
            img = img[..., :3]
        if img.dtype == np.uint16:
            img = (img.astype(np.float32) / 65535.0 * 255.0).clip(0, 255).astype(np.uint8)
        elif img.dtype != np.uint8:
            img = img.astype(np.uint8)
        return img

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        info = self.coco.imgs[img_id]
        img = self._load_image_uint8(info)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        masks, labels, boxes = [], [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            if w <= 0 or h <= 0:
                continue
            m = mask_utils.decode(a["segmentation"])
            if m.sum() == 0:
                continue
            masks.append(m)
            labels.append(int(a["category_id"]))

            boxes.append([float(x), float(y), float(x + w), float(y + h)])

        if len(masks) == 0:
            return self._empty_sample(img_id, info, img)

        if self.transforms is not None:
            transformed = self.transforms(
                image=img,
                masks=masks,
                bboxes=boxes,
                labels=labels,
            )
            img = transformed["image"]
            masks = transformed["masks"]
            labels = transformed["labels"]

        kept_masks, kept_labels, kept_boxes = [], [], []
        for m, lbl in zip(masks, labels):
            m = np.asarray(m)
            area = int(m.sum())
            if area < MIN_AREA_AFTER_AUGMENTATION:
                continue
            ys, xs = np.where(m > 0)
            x1, y1 = float(xs.min()), float(ys.min())
            x2, y2 = float(xs.max()) + 1.0, float(ys.max()) + 1.0
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            kept_masks.append(m.astype(np.uint8))
            kept_labels.append(int(lbl))
            kept_boxes.append([x1, y1, x2, y2])

        if len(kept_masks) == 0:
            return self._empty_sample(img_id, info, img)

        H, W = img.shape[:2]
        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1).contiguous()

        boxes_t = torch.as_tensor(kept_boxes, dtype=torch.float32)
        labels_t = torch.as_tensor(kept_labels, dtype=torch.int64)
        masks_t = torch.as_tensor(np.stack(kept_masks, axis=0), dtype=torch.uint8)
        areas_t = (boxes_t[:, 2] - boxes_t[:, 0]) * (boxes_t[:, 3] - boxes_t[:, 1])
        iscrowd_t = torch.zeros((len(kept_masks),), dtype=torch.int64)

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "masks": masks_t,
            "image_id": torch.tensor([img_id], dtype=torch.int64),
            "area": areas_t,
            "iscrowd": iscrowd_t,
        }
        return img_t, target

    def _empty_sample(self, img_id, info, img):
        H, W = info["height"], info["width"]
        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2, 0, 1).contiguous()
        target = {
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.zeros((0,), dtype=torch.int64),
            "masks": torch.zeros((0, img_t.shape[1], img_t.shape[2]), dtype=torch.uint8),
            "image_id": torch.tensor([img_id], dtype=torch.int64),
            "area": torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.zeros((0,), dtype=torch.int64),
        }
        return img_t, target


def collate_fn(batch):
    return tuple(zip(*batch))

In [9]:
def build_model(num_classes: int, pretrained: bool = True) -> nn.Module:

    weights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(weights=weights)

    in_feats = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_feats, num_classes)

    in_feats_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels

    # print(f"In layers = {in_feats_mask}")

    hidden = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_feats_mask, hidden, num_classes)

    return model


def _encode_mask(binary_mask: np.ndarray) -> dict:
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle


@torch.no_grad()
def evaluate_ap50(model, data_loader, device, score_thresh: float = 0.0):

    model.eval()

    coco_gt = data_loader.dataset.coco
    results = []

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        outputs = model(images)
        for tgt, out in zip(targets, outputs):
            img_id = int(tgt["image_id"].item())
            boxes = out["boxes"].cpu().numpy()
            scores = out["scores"].cpu().numpy()
            labels = out["labels"].cpu().numpy()


            masks = (out["masks"] > 0.5).squeeze(1).cpu().numpy().astype(np.uint8)

            for i in range(len(scores)):
                if scores[i] < score_thresh:
                    continue
                x1, y1, x2, y2 = boxes[i]
                w, h = float(x2 - x1), float(y2 - y1)
                if w <= 0 or h <= 0:
                    continue
                results.append({
                    "image_id": img_id,
                    "category_id": int(labels[i]),
                    "bbox": [float(x1), float(y1), w, h],
                    "score": float(scores[i]),
                    "segmentation": _encode_mask(masks[i]),
                })

    if len(results) == 0:
        return {"bbox_AP50": 0.0, "segm_AP50": 0.0, "num_predictions": 0}

    metrics = {}
    for iou_type in ("bbox", "segm"):
        coco_dt = coco_gt.loadRes(copy.deepcopy(results))
        coco_eval = COCOeval(coco_gt, coco_dt, iouType=iou_type)
        coco_eval.params.imgIds = sorted(coco_gt.imgs.keys())

        buf = io.StringIO()
        with redirect_stdout(buf):
            coco_eval.evaluate()
            coco_eval.accumulate()
            coco_eval.summarize()

        metrics[f"{iou_type}_AP"] = float(coco_eval.stats[0])
        metrics[f"{iou_type}_AP50"] = float(coco_eval.stats[1])

    metrics["num_predictions"] = len(results)
    return metrics

In [ ]:
def make_lr_lambda(warmup_iters, total_iters, milestones, gamma):

    milestone_iters = [int(m * total_iters) for m in milestones]

    def lr_lambda(it):
        if it < warmup_iters:
            return (it + 1) / max(1, warmup_iters)
        factor = 1.0
        for m in milestone_iters:
            if it >= m:
                factor *= gamma
        return factor

    return lr_lambda



def train_one_epoch(model, optimizer, scheduler, loader, device, epoch, log_every=20, grad_clip=10.0):
    model.train()
    running = 0.0
    n_seen = 0
    t0 = time.time()
    for it, (images, targets) in enumerate(loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())

        if not math.isfinite(loss.item()):
            print(f"  [epoch {epoch} it {it}] non-finite loss, skipping; components: "
                  + ", ".join(f"{k}={v.item():.3f}" for k, v in loss_dict.items()))
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        scheduler.step()

        running += loss.item()
        n_seen += 1
        if (it + 1) % log_every == 0:
            comp = "  ".join(f"{k.replace('loss_', '')}:{v.item():.3f}"
                             for k, v in loss_dict.items())
            print(f"  epoch {epoch}  iter {it+1}/{len(loader)}  "
                  f"loss={running/n_seen:.4f}  lr={optimizer.param_groups[0]['lr']:.2e}  [{comp}]")
    dt = time.time() - t0
    return running / max(1, n_seen), dt


def main():
  train_images = Path("/content/hw3-data-release/train_images")
  train_ann = Path("/content/hw3-data-release/annotations/train.json")
  val_ann = Path("/content/hw3-data-release/annotations/val.json")
  output_dir = Path("/content/drive/MyDrive/Visual Recognition/hw3")

  eval_every = 1
  seed = 2


  epochs = 15

  torch.manual_seed(seed)

  out = Path(output_dir)
  out.mkdir(parents=True, exist_ok=True)

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Device: {device}")
  if device.type == "cuda":
      print(f"  GPU: {torch.cuda.get_device_name(0)}")

  train_ds = CocoMyDataset(train_images, train_ann)
  val_ds = CocoMyDataset(train_images, val_ann)
  print(f"Train: {len(train_ds)} images   Val: {len(val_ds)} images")

  train_loader = DataLoader(
      train_ds, batch_size=2, shuffle=True,
      num_workers=4, collate_fn=collate_fn, pin_memory=True,
  )
  val_loader = DataLoader(
      val_ds, batch_size=1, shuffle=False,
      num_workers=4, collate_fn=collate_fn, pin_memory=True,
  )

  model = build_model(num_classes=5, pretrained=True)
  model.to(device)
  n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

  params = [p for p in model.parameters() if p.requires_grad]
  optimizer = torch.optim.SGD(
      params, lr=0.005, momentum=0.9, weight_decay=1e-4,
  )

  iters_per_epoch = len(train_loader)
  total_iters = iters_per_epoch * epochs
  warmup = min(500, total_iters // 4)
  scheduler = LambdaLR(optimizer, make_lr_lambda(warmup, total_iters, [0.8, 0.9], 0.1))

  best_segm_ap50 = -1.0
  for epoch in range(1, epochs + 1):
      print(f"\n=== Epoch {epoch}/{epochs} ===")
      avg_loss, dt = train_one_epoch(model, optimizer, scheduler, train_loader, device, epoch)
      print(f"  train_loss={avg_loss:.4f}   epoch_time={dt:.1f}s")

      record = {"epoch": epoch, "train_loss": avg_loss, "epoch_time_s": dt}

      if epoch % eval_every == 0 or epoch == epochs:
          metrics = evaluate_ap50(model, val_loader, device)
          print(f"  val: bbox_AP50={metrics['bbox_AP50']:.4f}  "
                f"segm_AP50={metrics['segm_AP50']:.4f}  "
                f"(bbox_AP={metrics['bbox_AP']:.4f}  segm_AP={metrics['segm_AP']:.4f})  "
                f"preds={metrics['num_predictions']}")
          record.update(metrics)

          if metrics["segm_AP50"] > best_segm_ap50:
              best_segm_ap50 = metrics["segm_AP50"]
              ckpt_path = out / "best.pt"
              torch.save({
                  "epoch": epoch,
                  "model": model.state_dict(),
                  "metrics": metrics,
              }, ckpt_path)
              print(f"  -> new best segm_AP50={best_segm_ap50:.4f}, saved to {ckpt_path}")

      torch.save({
          "epoch": epoch,
          "model": model.state_dict(),
      }, out / "last.pt")


if __name__ == "__main__":
    main()

In [ ]:

TTA_TRANSFORMS = [
    dict(hflip=False, vflip=False, transpose=False),  # identity
    dict(hflip=True,  vflip=False, transpose=False),  # horizontal flip
    dict(hflip=False, vflip=True,  transpose=False),  # vertical flip
    dict(hflip=True,  vflip=True,  transpose=False),  # 180° rotation
]

TTA_NMS_IOU = 0.5


def load_image(path: Path) -> torch.Tensor:
    img = tifffile.imread(str(path))
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.shape[-1] == 4:
        img = img[..., :3]
    if img.dtype == np.uint16:
        img = img.astype(np.float32) / 65535.0 * 255.0
    else:
        img = img.astype(np.float32)
    img = img / 255.0
    img = torch.from_numpy(img).permute(2, 0, 1).contiguous().float()
    return img


def encode_mask(binary_mask: np.ndarray) -> dict:
    rle = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("ascii")
    return rle

def augment_image(img: torch.Tensor, cfg: dict) -> torch.Tensor:
    if cfg["transpose"]:
        img = img.permute(0, 2, 1)

    if cfg["hflip"]:
        img = torch.flip(img, dims=[2])

    if cfg["vflip"]:
        img = torch.flip(img, dims=[1])

    return img


def deaugment_boxes(boxes: np.ndarray, cfg: dict, H: int, W: int) -> np.ndarray:

    if len(boxes) == 0:
        return boxes
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]

    if cfg["hflip"]:
        x1, x2 = W - boxes[:, 2], W - boxes[:, 0]
    if cfg["vflip"]:
        y1, y2 = H - boxes[:, 3], H - boxes[:, 1]
    if cfg["transpose"]:
        x1, y1, x2, y2 = y1.copy(), x1.copy(), y2.copy(), x2.copy()

    return np.stack([x1, y1, x2, y2], axis=1)


def deaugment_mask(mask: np.ndarray, cfg: dict) -> np.ndarray:
    if cfg["vflip"]:
        mask = mask[::-1, :]
    if cfg["hflip"]:
        mask = mask[:, ::-1]
    if cfg["transpose"]:
        mask = mask.T
    return np.ascontiguousarray(mask)

@torch.no_grad()
def predict_with_tta(
    model,
    img: torch.Tensor,
    device: torch.device,
    score_thresh: float,
    mask_thresh: float,
    max_dets_per_aug: int | None,
    tta_nms_iou: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:

    H, W = img.shape[1], img.shape[2]

    all_boxes  = []
    all_scores = []
    all_labels = []
    all_masks  = []

    for cfg in TTA_TRANSFORMS:
        aug_img = augment_image(img.clone(), cfg)
        out = model([aug_img.to(device)])[0]

        boxes_a      = out["boxes"].cpu().numpy()
        scores_a     = out["scores"].cpu().numpy()
        labels_a     = out["labels"].cpu().numpy()
        masks_soft_a = out["masks"].cpu().numpy()

        aug_H, aug_W = aug_img.shape[1], aug_img.shape[2]
        masks_bin_a  = (masks_soft_a >= mask_thresh).squeeze(1).astype(np.uint8)

        if max_dets_per_aug is not None and len(scores_a) > max_dets_per_aug:
            keep = np.argsort(-scores_a)[:max_dets_per_aug]
            boxes_a, scores_a, labels_a, masks_bin_a = (
                boxes_a[keep], scores_a[keep], labels_a[keep], masks_bin_a[keep]
            )

        keep_score = scores_a >= score_thresh
        if not keep_score.any():
            continue
        boxes_a, scores_a, labels_a, masks_bin_a = (
            boxes_a[keep_score], scores_a[keep_score],
            labels_a[keep_score], masks_bin_a[keep_score],
        )

        boxes_orig = deaugment_boxes(boxes_a, cfg, aug_H, aug_W)

        masks_orig = np.stack(
            [deaugment_mask(m, cfg) for m in masks_bin_a], axis=0
        )

        all_boxes.append(boxes_orig)
        all_scores.append(scores_a)
        all_labels.append(labels_a)
        all_masks.append(masks_orig)

    if not all_boxes:
        empty = np.zeros((0,), dtype=np.float32)
        return np.zeros((0, 4), dtype=np.float32), empty, empty.astype(np.int64), np.zeros((0, H, W), dtype=np.uint8)

    all_boxes  = np.concatenate(all_boxes,  axis=0)
    all_scores = np.concatenate(all_scores, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_masks  = np.concatenate(all_masks,  axis=0)

    boxes_t  = torch.from_numpy(all_boxes).float()
    scores_t = torch.from_numpy(all_scores).float()
    labels_t = torch.from_numpy(all_labels).long()

    keep_idx = ops.batched_nms(boxes_t, scores_t, labels_t, tta_nms_iou)
    keep_idx = keep_idx.numpy()

    return (
        all_boxes[keep_idx],
        all_scores[keep_idx],
        all_labels[keep_idx],
        all_masks[keep_idx],
    )



def main():

  test_images = Path("/content/hw3-data-release/test_release")
  test_meta   = Path("/content/hw3-data-release/test_image_name_to_ids.json")

  checkpoint = Path("/content/drive/MyDrive/Visual Recognition/hw3/best.pt")
  output     = Path("/content/drive/MyDrive/Visual Recognition/hw3/test-results.json")

  score_thresh      = 0.05
  mask_thresh       = 0.5
  max_dets          = None
  max_dets_per_aug  = None

  with open(test_meta) as f:
      meta = json.load(f)
  print(f"Loaded {len(meta)} test image entries from {test_meta}")
  print(f"TTA variants: {len(TTA_TRANSFORMS)}  configs: {TTA_TRANSFORMS}")

  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"Device: {device}")

  model = build_model(num_classes=5, pretrained=False)
  ckpt  = torch.load(checkpoint, map_location=device, weights_only=False)
  state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
  missing, unexpected = model.load_state_dict(state, strict=True)
  print(f"Loaded checkpoint: {checkpoint}")
  if isinstance(ckpt, dict) and "metrics" in ckpt:
      print(f"  checkpoint val metrics: {ckpt['metrics']}")

  model.to(device).eval()

  results    = []
  n_missing  = 0
  n_dropped_score = 0
  n_dropped_box   = 0

  for i, entry in enumerate(meta):
      fname      = entry["file_name"]
      image_id   = int(entry["id"])
      expected_h = int(entry["height"])
      expected_w = int(entry["width"])

      img_path = Path(test_images) / fname
      if not img_path.exists():
          print(f"  [skip] missing: {fname}")
          n_missing += 1
          continue

      img      = load_image(img_path)
      actual_h, actual_w = img.shape[1], img.shape[2]
      if (actual_h, actual_w) != (expected_h, expected_w):
          print(f"  [warn] {fname} actual ({actual_h},{actual_w}) != meta ({expected_h},{expected_w})")

      boxes, scores, labels, masks_bin = predict_with_tta(
          model, img, device,
          score_thresh=score_thresh,
          mask_thresh=mask_thresh,
          max_dets_per_aug=max_dets_per_aug,
          tta_nms_iou=TTA_NMS_IOU,
      )

      for j in range(len(scores)):
          if scores[j] < score_thresh:
              n_dropped_score += 1
              continue
          x1, y1, x2, y2 = boxes[j]
          w, h = float(x2 - x1), float(y2 - y1)
          if w <= 0 or h <= 0:
              n_dropped_box += 1
              continue
          m = masks_bin[j]
          if m.sum() == 0:
              n_dropped_box += 1
              continue
          rle = encode_mask(m)
          results.append({
              "image_id":    image_id,
              "bbox":        [float(x1), float(y1), w, h],
              "score":       float(scores[j]),
              "category_id": int(labels[j]),
              "segmentation": rle,
          })

      if (i + 1) % 25 == 0 or (i + 1) == len(meta):
          print(f"  [{i+1}/{len(meta)}] {fname}  preds_kept={len(results)}")

  results.sort(key=lambda r: (r["image_id"], -r["score"]))

  out_path = Path(output)
  out_path.parent.mkdir(parents=True, exist_ok=True)
  with open(out_path, "w") as f:
      json.dump(results, f)

  print(f"\nWrote: {out_path}")

if __name__ == "__main__":
    main()

